# FilamentIQ: print failure classifier

Trains a small image classifier (`failure` / `normal`) that runs in the browser every second.
The web demo uses it as a cheap first check and only asks Gemini to confirm when it sees a failure.

Run in Google Colab with a GPU: **Runtime > Change runtime type > T4 GPU**. Training takes about 10-20 minutes.

Before you start, add your Roboflow API key as a Colab secret named `ROBOFLOW_API_KEY`
(key icon in the left bar). Get the key from roboflow.com > Settings > API Keys (free account).

In [ ]:
!pip -q install ultralytics==8.4.220 roboflow==1.7.4 onnx onnxruntime
!git clone -q -b feat/ml-detection https://github.com/HDSH-Dharma/CloudComputing repo  # use main after the PR is merged
import sys
from pathlib import Path
sys.path.insert(0, "repo/ml")
import prep

## 1. Public datasets (real photos, checked by hand on Roboflow Universe)

Detection datasets: an image with a spaghetti box becomes `failure`; images with only stringing or no box are skipped.
`3d print detecting_V2` is a classification set from a front webcam and gives most of the `normal` images.

All are CC BY 4.0: cite them in the report (each Roboflow page has a BibTeX block).
`3Dprinting` (public domain, 276 photos) has no published version: fork it into your workspace on Roboflow,
click Generate, then add it to `DETECTION` with your workspace name.

In [ ]:
from google.colab import userdata
from roboflow import Roboflow

rf = Roboflow(api_key=userdata.get("ROBOFLOW_API_KEY"))

def download(workspace, project, fmt):
    p = rf.workspace(workspace).project(project)
    v = max(p.versions(), key=lambda v: int(v.version))
    return Path(v.download(fmt, location=f"raw/{project}", overwrite=True).location)

FAIL_CLASSES = {"spaghetti", "fail_spaghetti", "fail"}
DETECTION = [
    ("3d-printing-failure", "spaghetti-3d"),
    ("hnnm", "3d-printing-defects-eavhn"),
    ("bu-engme500-vision-project", "fdm-failures-spaghetti"),
    # ("<your-workspace>", "3dprinting-uo6du"),  # after forking, see above
]
items = []
for ws, proj in DETECTION:
    items += prep.from_detection(download(ws, proj, "yolov8"), FAIL_CLASSES)
items += prep.from_folders(download("robot-project-nmclh", "3d-print-detecting_v2-fxxli", "folder"),
                           {"Failed": "failure", "succeed": "normal"})
print(len(items), "public images")

## 2. Your own printer (strongly recommended)

Public photos come from other printers and cameras. A few videos from your own camera fix most of that gap.

Upload videos into `own_videos/normal/` and `own_videos/failure/` (Files panel, left bar).
Each video must show one class only: cut a failed print from the moment the failure is visible.
Frames are taken once per second. Skip this cell if you have no videos yet.

Keep the three demo videos (`demo-gagal-spaghetti`, `demo-sukses`, `demo-macet`) out of training:
they are the end-to-end test in the web demo.

In [ ]:
import subprocess
for video in sorted(Path("own_videos").glob("*/*")):
    out = Path("own_frames") / video.parent.name
    out.mkdir(parents=True, exist_ok=True)
    subprocess.run(["ffmpeg", "-loglevel", "error", "-y", "-i", str(video), "-vf", "fps=1",
                    str(out / f"{video.stem}_f%04d.jpg")], check=True)
if Path("own_frames").exists():
    own = prep.from_folders(Path("own_frames"), {"normal": "normal", "failure": "failure"})
    items += own
    print(len(own), "own frames")

## 3. Build train / val / test (split per photo or video, balanced)

In [ ]:
counts = prep.build(items, Path("data"))
for split, c in counts.items():
    print(f"{split:5}  failure {c['failure']:5}  normal {c['normal']:5}")

## 4. Train

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11n-cls.pt")  # pretrained on ImageNet, 1.5M parameters
model.train(data="data", epochs=30, imgsz=224, patience=10, project="runs", name="cls", exist_ok=True)

## 5. Test (images the model never saw)

Put these numbers in the report. Also look at the confusion matrix: a failure the model calls `normal` is a missed
alarm, a `normal` image called `failure` only costs one extra Gemini call.

In [ ]:
from IPython.display import Image, display

best = YOLO("runs/cls/weights/best.pt")
metrics = best.val(data="data", split="test", project="runs", name="test", exist_ok=True)
print("top-1 accuracy on test:", round(metrics.top1, 3))
for png in sorted(Path("runs/test").glob("confusion_matrix*.png")):
    display(Image(str(png), width=420))

## 6. Export for the browser

Put both files in `apps/web/public/models/` and restart `pnpm dev`. The video demo picks them up automatically.

In [ ]:
import json, shutil
from google.colab import files

shutil.copy(best.export(format="onnx", imgsz=224), "print-cls.onnx")
Path("print-cls.json").write_text(json.dumps({"names": [best.names[i] for i in range(len(best.names))], "imgsz": 224}))
files.download("print-cls.onnx")
files.download("print-cls.json")